# 08 — Stacking Ensemble (XGB + LGBM + CatBoost + FT-Transformer -> Ridge)

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, joblib, torch
from src.data import load_processed
from src.models.ft_transformer import FTTransformer
from src.models.stacker import StackingRegressor
from src.train import predict_torch
from src.evaluate import regression_metrics
from src.config import MODELS_DIR

In [ ]:
X_train, y_train = load_processed('train')
X_val, y_val = load_processed('val')
X_test, y_test = load_processed('test')

In [ ]:
xgb = joblib.load(MODELS_DIR / 'XGBoost_tuned.joblib') if (MODELS_DIR / 'XGBoost_tuned.joblib').exists() else joblib.load(MODELS_DIR / 'XGBoost.joblib')
lgbm = joblib.load(MODELS_DIR / 'LightGBM.joblib')
cb   = joblib.load(MODELS_DIR / 'CatBoost.joblib')
ftt = FTTransformer(X_train.shape[1])
ftt.load_state_dict(torch.load(MODELS_DIR / 'ft_transformer.pt', map_location='cpu'))

In [ ]:
base_train = {
    'xgb': xgb.predict(X_train), 'lgbm': lgbm.predict(X_train),
    'cb': cb.predict(X_train), 'ftt': predict_torch(ftt, X_train)
}
base_test = {
    'xgb': xgb.predict(X_test), 'lgbm': lgbm.predict(X_test),
    'cb': cb.predict(X_test), 'ftt': predict_torch(ftt, X_test)
}
stk = StackingRegressor(base_train, y_train, alpha=1.0)
preds = stk.predict(base_test)
print(regression_metrics(y_test, preds))
joblib.dump(stk, MODELS_DIR / 'stacker.joblib')